In [ ]:
import pandas as pd

# The dataset doesn't have a header row, so we assign column names manually
columns = ['target', 'ids', 'date', 'flag', 'user', 'text']

# Load the file
df = pd.read_csv('training.1600000.processed.noemoticon.csv')

# Preview the data
print(df.head())

   0  1467810369  Mon Apr 06 22:19:45 PDT 2009  NO_QUERY _TheSpecialOne_  \
0  0  1467810672  Mon Apr 06 22:19:49 PDT 2009  NO_QUERY   scotthamilton   
1  0  1467810917  Mon Apr 06 22:19:53 PDT 2009  NO_QUERY        mattycus   
2  0  1467811184  Mon Apr 06 22:19:57 PDT 2009  NO_QUERY         ElleCTF   
3  0  1467811193  Mon Apr 06 22:19:57 PDT 2009  NO_QUERY          Karoli   
4  0  1467811372  Mon Apr 06 22:20:00 PDT 2009  NO_QUERY        joy_wolf   

  @switchfoot http://twitpic.com/2y1zl - Awww, that's a bummer.  You shoulda got David Carr of Third Day to do it. ;D  
0  is upset that he can't update his Facebook by ...                                                                   
1  @Kenichan I dived many times for the ball. Man...                                                                   
2    my whole body feels itchy and like its on fire                                                                    
3  @nationwideclass no, it's not behaving at all....           

In [ ]:
# Reload the dataset using the 'python' engine to handle messy data
columns = ['target', 'ids', 'date', 'flag', 'user', 'text']

# The 'engine="python"' parameter handles those tokenizing errors
# 'on_bad_lines="skip"' will simply skip the few rows that are broken
df = pd.read_csv('training.1600000.processed.noemoticon.csv',
                 encoding='latin-1',
                 names=columns,
                 engine='python',
                 on_bad_lines='skip')

# Now apply the cleaning function
import re

def clean_text(text):
    text = str(text)
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text

df['cleaned_text'] = df['text'].apply(clean_text)

# Verify the output
print(df.head())

   target         ids                          date      flag  \
0       0  1467810369  Mon Apr 06 22:19:45 PDT 2009  NO_QUERY   
1       0  1467810672  Mon Apr 06 22:19:49 PDT 2009  NO_QUERY   
2       0  1467810917  Mon Apr 06 22:19:53 PDT 2009  NO_QUERY   
3       0  1467811184  Mon Apr 06 22:19:57 PDT 2009  NO_QUERY   
4       0  1467811193  Mon Apr 06 22:19:57 PDT 2009  NO_QUERY   

              user                                               text  \
0  _TheSpecialOne_  @switchfoot http://twitpic.com/2y1zl - Awww, t...   
1    scotthamilton  is upset that he can't update his Facebook by ...   
2         mattycus  @Kenichan I dived many times for the ball. Man...   
3          ElleCTF    my whole body feels itchy and like its on fire    
4           Karoli  @nationwideclass no, it's not behaving at all....   

                                        cleaned_text  
0     a thats a bummer  you shoulda got david car...  
1  is upset that he cant update his facebook by t...  
2   i

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# 1. Initialize the vectorizer
# We'll use max_features=10000 to keep it manageable and fast
tfidf = TfidfVectorizer(max_features=10000)

# 2. Convert text to numbers
X = tfidf.fit_transform(df['cleaned_text'])
y = df['target']

# 3. Split into training and testing sets
# This allows us to train on one part and test accuracy on the other
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Vectorization complete!")
print("Training set shape:", X_train.shape)

Vectorization complete!
Training set shape: (555618, 10000)


In [ ]:
# Check unique values in the target column
print("Unique values in target column:", df['target'].unique())

# Check the counts of all values
print("Counts of each class:")
print(df['target'].value_counts())

Unique values in target column: [0]
Counts of each class:
target
0    694523
Name: count, dtype: int64


In [ ]:
# 1. Download and Extract the pristine dataset directly from Stanford University
# (-O overwrites any broken zip files, -o forces the extraction to replace the broken CSV)
!wget -q -O trainingandtestdata.zip http://cs.stanford.edu/people/alecmgo/trainingandtestdata.zip
!unzip -q -o trainingandtestdata.zip

import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

print("1. Loading perfect data...")
# 2. Load the data
columns = ['target', 'ids', 'date', 'flag', 'user', 'text']
df = pd.read_csv('training.1600000.processed.noemoticon.csv',
                 encoding='latin-1',
                 names=columns,
                 engine='python',
                 on_bad_lines='skip')

print("\n--- Data Check (Should see both 0 and 4!) ---")
print(df['target'].value_counts())

# 3. Clean Text
print("\n2. Cleaning text...")
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text

df['cleaned_text'] = df['text'].apply(clean_text)

# 4. Vectorize
print("3. Vectorizing text (this takes a moment)...")
tfidf = TfidfVectorizer(max_features=10000)
X = tfidf.fit_transform(df['cleaned_text'])
y = df['target']

# 5. Split Data
print("4. Splitting data...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 6. Train Model
print("5. Training the model...")
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# 7. Evaluate
y_pred = model.predict(X_test)
print("\n--- MODEL RESULTS ---")
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

1. Loading perfect data...

--- Data Check (Should see both 0 and 4!) ---
target
0    800000
4    800000
Name: count, dtype: int64

2. Cleaning text...
3. Vectorizing text (this takes a moment)...
4. Splitting data...
5. Training the model...

--- MODEL RESULTS ---
Accuracy: 0.794453125
              precision    recall  f1-score   support

           0       0.80      0.78      0.79    159494
           4       0.79      0.81      0.80    160506

    accuracy                           0.79    320000
   macro avg       0.79      0.79      0.79    320000
weighted avg       0.79      0.79      0.79    320000



In [ ]:
def predict_sentiment(custom_text):
    # 1. Clean the text using the exact same function from earlier
    cleaned = clean_text(custom_text)

    # 2. Convert it to numbers (Notice we use .transform, NOT .fit_transform!)
    vectorized = tfidf.transform([cleaned])

    # 3. Ask the model to predict
    prediction = model.predict(vectorized)[0]

    # 4. Translate the output (0 = Negative, 4 = Positive)
    if prediction == 4:
        return f"'{custom_text}' ---> 🟢 POSITIVE"
    else:
        return f"'{custom_text}' ---> 🔴 NEGATIVE"

# Let's test it out!
print(predict_sentiment("I absolutely love the new design, it works perfectly!"))
print(predict_sentiment("My order arrived three days late and the box was completely crushed."))
print(predict_sentiment("Just learned how to build an NLP pipeline today. Feeling great!"))

'I absolutely love the new design, it works perfectly!' ---> 🟢 POSITIVE
'My order arrived three days late and the box was completely crushed.' ---> 🔴 NEGATIVE
'Just learned how to build an NLP pipeline today. Feeling great!' ---> 🟢 POSITIVE


In [15]:
import pandas as pd

print("Preparing data for Tableau...")
df_sample = df.sample(n=5000, random_state=42).copy()

# ---> THE BULLETPROOF DATE FIX <---
# 1. Delete the confusing 'PDT' and 'PST' strings from the text
df_sample['date'] = df_sample['date'].astype(str).replace(r' (PDT|PST) ', ' ', regex=True)

# 2. Now that the timezone letters are gone, cleanly convert it to a standard format!
df_sample['date'] = pd.to_datetime(df_sample['date']).dt.strftime('%Y-%m-%d %H:%M:%S')

# Make the predictions
sample_vectorized = tfidf.transform(df_sample['cleaned_text'])
df_sample['predicted_label'] = model.predict(sample_vectorized)
df_sample['Sentiment'] = df_sample['predicted_label'].map({0: 'Negative', 4: 'Positive'})

# Export
final_export = df_sample[['date', 'user', 'text', 'Sentiment']]
final_export.to_csv('final_sentiment_data.csv', index=False)

print("✅ Date fixed! Export complete. Please download the new file.")
print(final_export[['date', 'Sentiment']].head())

Preparing data for Tableau...
✅ Date fixed! Export complete. Please download the new file.
                       date Sentiment
541200  2009-06-16 18:18:12  Positive
750     2009-04-06 23:11:14  Positive
766711  2009-06-23 13:40:11  Positive
285055  2009-06-01 10:26:07  Negative
705995  2009-06-20 12:56:51  Positive
